## Manual validation - Exploration of results

In [9]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent.resolve()))
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [10]:
from src.paths import VALIDATION_SAMPLE
df = pd.read_excel(VALIDATION_SAMPLE)

In [11]:
# Total unique topics and subtopics
print(len(df['topic'].unique()))
print(len(df['subtopics'].unique()))

21
87


In [12]:
def generate_invalid_topics_table(classified_path: str = VALIDATION_SAMPLE) -> pd.DataFrame:
    """
    Generate and print a table showing topics with invalid topic/subtopic counts.
    
    Columns:
    - Topic: Topic name
    - Invalids: Count of posts with invalid topic OR invalid subtopic
    - Total: Total count of posts for this topic
    - Difference: Total - Invalids (valid posts)
    
    Args:
        classified_path: Path to CLASSIFIED_POSTS CSV/Excel file
    
    Returns:
        DataFrame with the table
    """
    df = pd.read_excel(classified_path)
    
    df['is_invalid'] = (df['topic_veredict'] != True) | (df['subtopic_veredict'] != True)
    
    results = []
    
    for topic_name in df['topic'].unique():
        topic_df = df[df['topic'] == topic_name]
        
        total = len(topic_df)
        invalids = topic_df['is_invalid'].sum()
        difference = total - invalids
        
        results.append({
            'Topic': topic_name,
            'Invalids': int(invalids),
            'Total': total,
            'Difference': difference
        })

    result_df = pd.DataFrame(results)
    
    result_df = result_df.sort_values('Invalids', ascending=False).reset_index(drop=True)
    
    totals = {
        'Topic': 'Total',
        'Invalids': result_df['Invalids'].sum(),
        'Total': result_df['Total'].sum(),
        'Difference': result_df['Difference'].sum()
    }
    result_df = pd.concat([result_df, pd.DataFrame([totals])], ignore_index=True)
    

    print("=" * 100)
    print("Invalid topics table")
    print("=" * 100)
    print(result_df.to_string(index=False))
    print("=" * 100)
    
    return result_df


df_invalid = generate_invalid_topics_table()

Invalid topics table
                                                           Topic  Invalids  Total  Difference
                   OpenSSL installation/build and runtime errors        11     55          44
                       Block cipher modes, IV/nonce, and padding         8     21          13
                       Crypto library/API implementation in code         6     23          17
                       Application data encryption & key storage         4     12           8
                     File handling, integrity checks & checksums         3     15          12
      Public-key cryptography: encryption and digital signatures         2     18          16
       Client-server secure communication (HTTP/HTTPS, sessions)         2     14          12
    Key protection on devices (memory/disk access, threat model)         2     11           9
             Password hashing with salt (bcrypt, rainbow tables)         2     25          23
                   X.509 certificates, 

In [13]:
df.groupby('topic')['topic_veredict'].mean().sort_values(ascending=False)

topic
Application data encryption & key storage                           1.000000
Conceptual crypto explanations and learning                         1.000000
Encoding and byte/string conversions (Base64, binary)               1.000000
Cryptosystem design pitfalls & attack resistance                    1.000000
Cryptographic hash functions & collisions                           1.000000
X.509 certificates, signing, and trust chains                       1.000000
Security vs performance (cost, speed, hardware limits)              1.000000
Secure random number generation (entropy, seeding, PRNG)            1.000000
Key sizes and security parameters selection                         1.000000
Number theory for crypto (primes, factoring, modular arithmetic)    1.000000
User authentication and account management                          1.000000
Password hashing with salt (bcrypt, rainbow tables)                 1.000000
Public-key cryptography: encryption and digital signatures          0.

In [14]:
df['topic_veredict'].mean()

np.float64(0.9323943661971831)

In [15]:
df['subtopic_veredict'].mean()

np.float64(0.8760563380281691)

In [ ]:
def compute_kappa_segment(df_segment, pair_name):
    n = len(df_segment)
    pe = 0.50  

    topic_agreements = (
        df_segment['topic_validation_1'] == df_segment['topic_validation_2']
    ).sum()
    po_topic = topic_agreements / n
    kappa_topic = (po_topic - pe) / (1 - pe)

    subtopic_agreements = (
        df_segment['subtopic_validation_1'] == df_segment['subtopic_validation_2']
    ).sum()
    po_subtopic = subtopic_agreements / n
    kappa_subtopic = (po_subtopic - pe) / (1 - pe)

    print(f"\nResults — {pair_name}")
    print(f"Total records: {n}")
    print("-" * 40)
    print("TOPICS:")
    print(f"  - Observed agreement: {po_topic:.2%}")
    print(f"  - Kappa value: {kappa_topic:.3f}")
    print("\nSUBTOPICS:")
    print(f"  - Observed agreement: {po_subtopic:.2%}")
    print(f"  - Kappa value: {kappa_subtopic:.3f}")

    return {
        "pair": pair_name,
        "kappa_topic": kappa_topic,
        "kappa_subtopic": kappa_subtopic,
        "po_topic": po_topic,
        "po_subtopic": po_subtopic,
        "n": n
    }


def compute_kappa_metrics_pairs(df):
    # Pair 1: rows 1–177 
    df_pair_1 = df.iloc[:177]

    # Pair 2: rows 178–end
    df_pair_2 = df.iloc[177:]

    results = []
    results.append(compute_kappa_segment(df_pair_1, "Pair 1 (rows 1–177)"))
    results.append(compute_kappa_segment(df_pair_2, "Pair 2 (rows 178–end)"))

    return results


compute_kappa_metrics_pairs(df)


Results — Pair 1 (rows 1–177)
Total records: 177
----------------------------------------
TOPICS:
  - Observed agreement: 97.18%
  - Kappa value: 0.944

SUBTOPICS:
  - Observed agreement: 94.35%
  - Kappa value: 0.887

Results — Pair 2 (rows 178–end)
Total records: 178
----------------------------------------
TOPICS:
  - Observed agreement: 98.88%
  - Kappa value: 0.978

SUBTOPICS:
  - Observed agreement: 96.07%
  - Kappa value: 0.921


[{'pair': 'Pair 1 (rows 1–177)',
  'kappa_topic': np.float64(0.9435028248587571),
  'kappa_subtopic': np.float64(0.887005649717514),
  'po_topic': np.float64(0.9717514124293786),
  'po_subtopic': np.float64(0.943502824858757),
  'n': 177},
 {'pair': 'Pair 2 (rows 178–end)',
  'kappa_topic': np.float64(0.9775280898876404),
  'kappa_subtopic': np.float64(0.9213483146067416),
  'po_topic': np.float64(0.9887640449438202),
  'po_subtopic': np.float64(0.9606741573033708),
  'n': 178}]